# RAG Full Pipeline Execution & Idempotency Analysis

This notebook demonstrates the end-to-end RAG ingestion pipeline and explains the "Second Execution" problem where successful results can be overwritten by subsequent runs.

## 1. Setup and Initialization

We initialize the `RAGIngestion` service which handles PDF parsing, chunking, BigQuery staging, and BQML vectorization.

In [1]:
import sys
import os
import json
from google.cloud import bigquery

# Ensure project root is in path
sys.path.append("../../..")

from pipelines.enterprise_knowledge_base.app.rag_ingestion.pipeline import RAGIngestion
from pipelines.enterprise_knowledge_base.app.rag_ingestion.schemas import IngestDocumentRequest

# Configuration
PROJECT_ID = "ag-core-dev-fdx7"
os.environ["PROJECT_ID"] = PROJECT_ID
GCS_URI = "gs://kb-it/client-confidential/orchestration-test/tester/SoW - Innovation.pdf"

rag_pipeline = RAGIngestion()
bq_client = bigquery.Client(project=PROJECT_ID)
print(f"RAG Pipeline initialized for project: {PROJECT_ID}")

2026-04-27 17:39:56.036 | INFO     | pipelines.enterprise_knowledge_base.app.rag_ingestion.pipeline:__init__:38 - Initialized RAGIngestion | CHUNK_SIZE: 1000 | OVERLAP: 100


RAG Pipeline initialized for project: ag-core-dev-fdx7


## 2. The "Second Execution" Problem Explained

The pipeline is designed to be **idempotent**. Every time it runs for a specific URI, it performs the following logic:

1.  **Clear existing chunks**: It deletes all rows in BigQuery matching the URI (`DELETE FROM ... WHERE gcs_uri = ...`).
2.  **Ingest new chunks**: It parses the PDF and inserts new rows.
3.  **Vectorize**: It runs an `UPDATE` query to generate embeddings for the *new* rows.

### The Pitfall
If you run the pipeline twice:
- **Run 1** finishes and correctly vectorizes 15 chunks.
- **Run 2** starts and immediately **deletes** those 15 vectorized chunks.
- If you look at BigQuery *after* Run 2 has inserted new rows but *before* it has finished vectorizing them, you will see **null embeddings**.

This explains why the timestamps in BigQuery might be newer than the "Success" logs you saw—you were likely looking at the results of a second, incomplete run.

## 3. Full Pipeline Execution

We will now execute the `run()` method, which orchestrates the entire flow (Ingest -> Stage -> Vectorize).

In [2]:
print(f"Executing full pipeline for: {GCS_URI}")

req = IngestDocumentRequest(gcs_uri=GCS_URI)
resp = rag_pipeline.run(req)

print(f"\n--- Execution Results ---")
print(f"Status: {resp.execution_status}")
print(f"Chunks created: {resp.chunk_count}")
print(f"Processed URI: {resp.processed_uri}")

2026-04-27 17:39:56.453 | INFO     | pipelines.enterprise_knowledge_base.app.rag_ingestion.pipeline:run:57 - Starting end-to-end pipeline for: gs://kb-it/client-confidential/orchestration-test/tester/SoW - Innovation.pdf
2026-04-27 17:39:56.454 | INFO     | pipelines.enterprise_knowledge_base.app.rag_ingestion.pipeline:ingest_document:87 - Starting ingestion for document: gs://kb-it/client-confidential/orchestration-test/tester/SoW - Innovation.pdf


Executing full pipeline for: gs://kb-it/client-confidential/orchestration-test/tester/SoW - Innovation.pdf


2026-04-27 17:39:58.971 | INFO     | pipelines.enterprise_knowledge_base.app.rag_ingestion.pipeline:_clear_existing_chunks:276 - Cleared 15 existing chunks for: gs://kb-it/client-confidential/orchestration-test/tester/SoW - Innovation.pdf
2026-04-27 17:39:58.972 | INFO     | pipelines.enterprise_knowledge_base.app.rag_ingestion.pipeline:_copy_to_staging:367 - Staging document: gs://kb-it/client-confidential/orchestration-test/tester/SoW - Innovation.pdf -> gs://ag-core-dev-fdx7-rag-staging/ingested/6568a8e3/SoW - Innovation.pdf
2026-04-27 17:40:04.727 | INFO     | pipelines.enterprise_knowledge_base.app.rag_ingestion.pipeline:ingest_document:110 - Successfully staged 15 chunks to BigQuery.
2026-04-27 17:40:04.727 | DEBUG    | pipelines.enterprise_knowledge_base.app.rag_ingestion.pipeline:_move_blob_to_processed:401 - Moving staging file ingested/6568a8e3/SoW - Innovation.pdf to processed/6568a8e3/SoW - Innovation.pdf
2026-04-27 17:40:05.314 | INFO     | pipelines.enterprise_knowledge_b


--- Execution Results ---
Status: SUCCESS
Chunks created: 15
Processed URI: gs://kb-it/client-confidential/orchestration-test/tester/SoW - Innovation.pdf


## 4. Detailed BigQuery Verification

Now we verify that the embeddings were actually generated and are present in the table. We check for `vectorized_at` and the length of the `embedding` array.

In [3]:
query = f"""
    SELECT 
        chunk_id, 
        created_at, 
        vectorized_at, 
        ARRAY_LENGTH(embedding) as embed_len
    FROM `{PROJECT_ID}.knowledge_base.documents_chunks` 
    WHERE NORMALIZE(gcs_uri) = NORMALIZE(@uri)
    ORDER BY created_at DESC
"""

job_config = bigquery.QueryJobConfig(
    query_parameters=[bigquery.ScalarQueryParameter("uri", "STRING", GCS_URI)]
)

print(f"Verifying BigQuery data for: {GCS_URI}\n")
results = bq_client.query(query, job_config=job_config).result()

rows_found = 0
vectorized_count = 0

for row in results:
    rows_found += 1
    if row.vectorized_at is not None and row.embed_len > 0:
        vectorized_count += 1
    
    # Print first few rows as example
    if rows_found <= 3:
        print(f"Chunk ID: {row.chunk_id[:8]}... | Created: {row.created_at} | Vectorized: {row.vectorized_at} | Embed Len: {row.embed_len}")

print(f"\nSummary:")
print(f"- Total rows found: {rows_found}")
print(f"- Correctly vectorized: {vectorized_count}")

if vectorized_count == rows_found and rows_found > 0:
    print("\n✅ SUCCESS: All chunks are fully vectorized in BigQuery!")
else:
    print("\n❌ FAILURE: Some chunks are missing embeddings.")

Verifying BigQuery data for: gs://kb-it/client-confidential/orchestration-test/tester/SoW - Innovation.pdf

Chunk ID: 81f5946a... | Created: 2026-04-27 17:40:00.862271+00:00 | Vectorized: 2026-04-27 17:40:06.897286+00:00 | Embed Len: 1408
Chunk ID: 9061e692... | Created: 2026-04-27 17:40:00.862257+00:00 | Vectorized: 2026-04-27 17:40:06.897286+00:00 | Embed Len: 1408
Chunk ID: 716eec0e... | Created: 2026-04-27 17:40:00.861085+00:00 | Vectorized: 2026-04-27 17:40:06.897286+00:00 | Embed Len: 1408

Summary:
- Total rows found: 15
- Correctly vectorized: 15

✅ SUCCESS: All chunks are fully vectorized in BigQuery!
